Calcul des descripteurs audio

In [ ]:
import os

from os import listdir
from os.path import isfile, join

import pandas as pd

from glob import glob

import librosa
import numpy as np

import matplotlib.pyplot as plt

from scipy.signal import hilbert, chirp

import scipy

import math

In [ ]:
#chargement des données du drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Chargement du fichier avec la description des sons
data_dir_ini = '/content/drive/MyDrive/Article sound/audio/4_norm/'
audio_files_ini = glob(data_dir_ini+'/*.wav')

df = pd.read_table("/content/drive/MyDrive/Article sound/data/data_ini_2digits.csv", sep=',' , encoding='utf-8-sig')


In [ ]:
df

,sound_sample_name,sample,form,geol,hit_number,sample_name,sample_number,sound_sample_name_simple,sound_number,mass_ini,...,mass_n,mass_end,f1_ini,a1_ini,f1_0,a1_0,f1_n,a1_n,f1_end,a1_end
0,L_pl_a_1,L,pl,a,1,S01,1,S01_1,1,3458.07,...,3481.78,3460.54,2748.0,0.3712,2590.0,0.7269,2499.756030,1.017641,2664.80616,0.637452
1,L_pl_a_2,L,pl,a,2,S01,1,S01_2,2,3458.07,...,3481.78,3460.54,2748.0,0.3712,2590.0,0.7269,2499.756030,1.017641,2664.80616,0.637452
2,L_pl_a_3,L,pl,a,3,S01,1,S01_3,3,3458.07,...,3481.78,3460.54,2748.0,0.3712,2590.0,0.7269,2499.756030,1.017641,2664.80616,0.637452
3,L_pl_a_4,L,pl,a,4,S01,1,S01_4,4,3458.07,...,3481.78,3460.54,2748.0,0.3712,2590.0,0.7269,2499.756030,1.017641,2664.80616,0.637452
4,L_pl_a_5,L,pl,a,5,S01,1,S01_5,5,3458.07,...,3481.78,3460.54,2748.0,0.3712,2590.0,0.7269,2499.756030,1.017641,2664.80616,0.637452
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,P_pl_a_5,P,pl,a,5,S31,31,S31_5,5,1636.50,...,1656.34,1636.62,3020.0,0.4311,2943.0,0.3937,2854.714907,0.373521,2906.21575,0.454652
161,P_pl_a_6,P,pl,a,6,S31,31,S31_6,6,1636.50,...,1656.34,1636.62,3020.0,0.4311,2943.0,0.3937,2854.714907,0.373521,2906.21575,0.454652
162,P_pl_a_7,P,pl,a,7,S31,31,S31_7,7,1636.50,...,1656.34,1636.62,3020.0,0.4311,2943.0,0.3937,2854.714907,0.373521,2906.21575,0.454652
163,P_pl_a_8,P,pl,a,8,S31,31,S31_8,8,1636.50,...,1656.34,1636.62,3020.0,0.4311,2943.0,0.3937,2854.714907,0.373521,2906.21575,0.454652


Calcul des descripteurs

In [ ]:
#Calcul du f1 audio
def compute_f1_audio(y, sr, n_fft, sound_name):
    # Compute F1 as the frequency with maximum intensity
    spectrum = abs(np.fft.fft(y))[0: int(len(y) / 2)]
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    f1 = freqs[np.argmax(spectrum)]

    # Plots
    plt.plot(freqs[:-1], spectrum)
    plt.title(sound_name)
    plt.legend(["F1: " + '%.f' % f1])
    plt.show()
    print(sound_name, '\t', '%.f' % f1)


    return(f1)

In [ ]:
#Calcul du coefficient d'attenuation c et du facteur de qualité théorique (à partir de f1_audio et du facteur d'atténuation de l'enveloppe)

#Fonction pour déterminer l'enveloppe
def find_peaks_window(x, y, window):
    peak_x = []
    peak_vals = []
    n = len(y)
    for i in range(window, n - window):
        if (y[i] > np.max(y[i-window:i])) and (y[i] > np.max(y[i+1:i+1+window])):
            peak_x.append(x[i])
            peak_vals.append(y[i])
    return np.array(peak_x), np.array(peak_vals)

#Fonction de l'enveloppe
def monoExp(x, m, c, b):
    return m * np.exp(-c * x) + b

#Fit de l'enveloppe
def compute_damping(y, sr, sound_name):
    time = np.arange(len(y)) / sr

    peak_x, peak_vals = find_peaks_window(time, y, window=20)

    m0 = 1
    b0 = 0
    c0 = 1.0 / (peak_x[1] - peak_x[0])

    # perform the fit
    p0 = (m0, c0, b0)
    params, cv = scipy.optimize.curve_fit(monoExp, peak_x, peak_vals, p0, maxfev=2000)
    m, c, b = params

    # determine quality of the fit
    squaredDiffs = np.square(peak_vals - monoExp(peak_x, m, c, b))
    squaredDiffsFromMean = np.square(peak_vals - np.mean(peak_vals))
    rSquared = 1 - np.sum(squaredDiffs) / np.sum(squaredDiffsFromMean)
    print(f"R² = {rSquared}")

    # print the parameters
    print(f"Y = {m} * e^(-{c} * x) + {b}")

    plt.title(sound_name)

    plt.plot(time, y, zorder=1)

    plt.scatter(peak_x, peak_vals, color='green', label="Pics détectés")
    plt.plot(peak_x, monoExp(peak_x, m, c, b), '--', label="fitted", linewidth=2, zorder=10)

    plt.legend(f"R² = {rSquared}")

    plt.show()

    return c

Calcul des descripteurs

In [ ]:
audio_files_ini.sort()

for file in range(0, len(audio_files_ini),1):

    path = audio_files_ini[file]

    name = os.path.basename(path)
    name_simple = name.replace("_norm.wav", "")

    y, sr = librosa.load(path, sr=None)
    frame_lenght = len(y)
    row_index = df.loc[(df == name_simple).any(axis=1)].index[0]

    print('------------------------')
    print('\n', name_simple)

    plt.plot(np.arange(len(y)) / sr, y)
    plt.title(name_simple)
    plt.show()



    df.loc[row_index, "sound_path"] = path

    rms = librosa.feature.rms(y=y, S=None, frame_length=frame_lenght, center=False)[0][0]
    df.loc[row_index, "rms"] = rms

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=frame_lenght, center=False, window='boxcar')[0][0]
    df.loc[row_index, "centroid"] = centroid

    flatness = librosa.feature.spectral_flatness(y=y, S=None, n_fft=frame_lenght, center=False, window='boxcar')[0][0]

    df.loc[row_index, "flatness"] = flatness

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, n_fft=frame_lenght, center=False, window='boxcar')[0][0]

    df.loc[row_index, "rolloff"] = rolloff

    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_fft=frame_lenght, center=False, n_bands=1, window='boxcar')[0][0]

    df.loc[row_index, "contrast"] = contrast

    f1 = compute_f1_audio(y, sr, frame_lenght, name_simple)

    df.loc[row_index, "f1_audio"] = f1

    c = compute_damping(y, sr, name_simple)
    df.loc[row_index, "c"] = c

    Q= (3.14 * f1 )/c
    df.loc[row_index, "Q"] = Q

    print("RMS: ", rms)
    print("Centroid: ", centroid)
    print("Flatness: ", flatness)
    print("Rolloff: ", rolloff)
    print("Contrast: ", contrast)
    print("F1: ", f1)
    print("c: ", c)
    print("Q: ", Q)







Output hidden; open in https://colab.research.google.com to view.

In [ ]:
df

,sound_sample_name,sample,form,geol,hit_number,sample_name,sample_number,sound_sample_name_simple,sound_number,mass_ini,...,a1_end,sound_path,rms,centroid,flatness,rolloff,contrast,f1_audio,c,Q
0,L_pl_a_1,L,pl,a,1,S01,1,S01_1,1,3458.07,...,0.637452,/content/drive/MyDrive/Article sound/audio/4_n...,0.122204,5841.668799,0.008963,12397.290432,14.673802,3223.793395,55.140871,183.579095
1,L_pl_a_2,L,pl,a,2,S01,1,S01_2,2,3458.07,...,0.637452,/content/drive/MyDrive/Article sound/audio/4_n...,0.129421,6028.623430,0.007643,14052.751905,10.566528,3227.942422,59.485896,170.388946
2,L_pl_a_3,L,pl,a,3,S01,1,S01_3,3,3458.07,...,0.637452,/content/drive/MyDrive/Article sound/audio/4_n...,0.111234,6446.748089,0.012436,16177.053345,12.863068,3223.793395,57.890860,174.858539
3,L_pl_a_4,L,pl,a,4,S01,1,S01_4,4,3458.07,...,0.637452,/content/drive/MyDrive/Article sound/audio/4_n...,0.128124,6010.692139,0.008403,13538.272650,14.208599,3227.942422,63.182687,160.419565
4,L_pl_a_5,L,pl,a,5,S01,1,S01_5,5,3458.07,...,0.637452,/content/drive/MyDrive/Article sound/audio/4_n...,0.130879,6893.026751,0.013390,16525.571550,19.863773,3227.942422,60.316329,168.043038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,P_pl_a_5,P,pl,a,5,S31,31,S31_5,5,1636.50,...,0.454652,/content/drive/MyDrive/Article sound/audio/4_n...,0.186941,6135.595823,0.003788,13310.076207,16.549231,3456.138865,26.039233,416.766354
161,P_pl_a_6,P,pl,a,6,S31,31,S31_6,6,1636.50,...,0.454652,/content/drive/MyDrive/Article sound/audio/4_n...,0.174533,5758.063079,0.003231,9463.928874,16.699080,3456.138865,26.775503,405.306146
162,P_pl_a_7,P,pl,a,7,S31,31,S31_7,7,1636.50,...,0.454652,/content/drive/MyDrive/Article sound/audio/4_n...,0.129668,4726.509981,0.000583,6634.292972,20.380640,3451.989839,26.909441,402.804655
163,P_pl_a_8,P,pl,a,8,S31,31,S31_8,8,1636.50,...,0.454652,/content/drive/MyDrive/Article sound/audio/4_n...,0.201567,5464.058606,0.002583,8414.225233,13.928079,3456.138865,24.830045,437.062279


Evaluation de l'endommagement et création des labels

In [ ]:
#Evaluation de l'endommagement et création des labels
for file in range(0, len(audio_files_ini),1):

    path = audio_files_ini[file]
    name = os.path.basename(path)
    name_simple = name.replace("_norm.wav", "")

    row_index = df.loc[(df == name_simple).any(axis=1)].index[0]

    #Endommagement final = 1-RDEMfrost
    df.loc[row_index, "D"] = 1-((df.loc[row_index, "f1_n"])*(df.loc[row_index, "f1_n"]))/((df.loc[row_index, "f1_0"])*(df.loc[row_index, "f1_0"]))

    #Facteur de qualité
    df.loc[row_index, "Q_vibration_initial"] = (1/(2*df.loc[row_index, "a1_0"]))*100   #Facteur de qualité initial

    #Pour D_01
    df.loc[row_index, "frost_resistant_01"] = 1 if df.loc[row_index, "D"] < 0.1 else 0

    #Pour D_001
    df.loc[row_index, "frost_resistant_001"] = 1 if df.loc[row_index, "D"] < 0.01 else 0


/tmp/ipykernel_18410/3471500268.py:11: RuntimeWarning: invalid value encountered in scalar divide
  df.loc[row_index, "D"] = 1-((df.loc[row_index, "f1_n"])*(df.loc[row_index, "f1_n"]))/((df.loc[row_index, "f1_0"])*(df.loc[row_index, "f1_0"]))
/tmp/ipykernel_18410/3471500268.py:14: RuntimeWarning: divide by zero encountered in scalar divide
  df.loc[row_index, "Q_vibration_initial"] = (1/(2*df.loc[row_index, "a1_0"]))*100   #Facteur de qualité initial


In [ ]:
pd.options.display.max_columns = None
df.head()

,sound_sample_name,sample,form,geol,hit_number,sample_name,sample_number,sound_sample_name_simple,sound_number,mass_ini,mass_0,mass_n,mass_end,f1_ini,a1_ini,f1_0,a1_0,f1_n,a1_n,f1_end,a1_end,sound_path,rms,centroid,flatness,rolloff,contrast,f1_audio,c,Q,D,Q_vibration_initial,frost_resistant_01,frost_resistant_001
0,L_pl_a_1,L,pl,a,1,S01,1,S01_1,1,3458.07,3481.07,3481.78,3460.54,2748.0,0.3712,2590.0,0.7269,2499.75603,1.017641,2664.80616,0.637452,/content/drive/MyDrive/Article sound/audio/4_n...,0.122204,5841.668799,0.008963,12397.290432,14.673802,3223.793395,55.140871,183.579095,0.068472,68.785252,1.0,0.0
1,L_pl_a_2,L,pl,a,2,S01,1,S01_2,2,3458.07,3481.07,3481.78,3460.54,2748.0,0.3712,2590.0,0.7269,2499.75603,1.017641,2664.80616,0.637452,/content/drive/MyDrive/Article sound/audio/4_n...,0.129421,6028.623430,0.007643,14052.751905,10.566528,3227.942422,59.485896,170.388946,0.068472,68.785252,1.0,0.0
2,L_pl_a_3,L,pl,a,3,S01,1,S01_3,3,3458.07,3481.07,3481.78,3460.54,2748.0,0.3712,2590.0,0.7269,2499.75603,1.017641,2664.80616,0.637452,/content/drive/MyDrive/Article sound/audio/4_n...,0.111234,6446.748089,0.012436,16177.053345,12.863068,3223.793395,57.890860,174.858539,0.068472,68.785252,1.0,0.0
3,L_pl_a_4,L,pl,a,4,S01,1,S01_4,4,3458.07,3481.07,3481.78,3460.54,2748.0,0.3712,2590.0,0.7269,2499.75603,1.017641,2664.80616,0.637452,/content/drive/MyDrive/Article sound/audio/4_n...,0.128124,6010.692139,0.008403,13538.272650,14.208599,3227.942422,63.182687,160.419565,0.068472,68.785252,1.0,0.0
4,L_pl_a_5,L,pl,a,5,S01,1,S01_5,5,3458.07,3481.07,3481.78,3460.54,2748.0,0.3712,2590.0,0.7269,2499.75603,1.017641,2664.80616,0.637452,/content/drive/MyDrive/Article sound/audio/4_n...,0.130879,6893.026751,0.013390,16525.571550,19.863773,3227.942422,60.316329,168.043038,0.068472,68.785252,1.0,0.0


Exportation des données par son en CSV

In [ ]:
# Exporter en CSV sans l’index
df.to_csv('/content/drive/MyDrive/Article sound/data/sound_features.csv', index=False, encoding='utf-8')

Exportation des données par échantillon

In [ ]:
frost_resistant_01 = (df.groupby("sample_name")["frost_resistant_01"]
          .apply(list)
          .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
          .add_prefix("frost_resitant_0"))

frost_resistant_001 = (df.groupby("sample_name")["frost_resistant_001"]
            .apply(list)
            .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
            .add_prefix("frost_resitant_00"))

form = (df.groupby("sample_name")["form"]
            .apply(list)
            .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
            .add_prefix("form_"))

f1_vibration = (df.groupby("sample_name")["f1_ini"]
          .apply(list)
          .apply(max)
          .to_frame(name="f1_vibration"))

centroid = (df.groupby("sample_name")["centroid"]
          .apply(list)
          .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
          .add_prefix("centroid_"))

rms = (df.groupby("sample_name")["rms"]
          .apply(list)
          .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
          .add_prefix("rms_"))

flatness = (df.groupby("sample_name")["flatness"]
            .apply(list)
            .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
            .add_prefix("flatness_"))

rolloff = (df.groupby("sample_name")["rolloff"]
          .apply(list)
          .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
          .add_prefix("rolloff_"))

contrast = (df.groupby("sample_name")["contrast"]
            .apply(list)
            .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
            .add_prefix("contrast_"))

f1_audio = (df.groupby("sample_name")["f1_audio"]
            .apply(list)
            .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
            .add_prefix("f1_audio_"))

c = (df.groupby("sample_name")["c"]
          .apply(list)
          .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
          .add_prefix("c_"))

Q = (df.groupby("sample_name")["Q"]
          .apply(list)
          .apply(lambda x: pd.Series(x, index=range(1, len(x)+1)))
          .add_prefix("Q_"))


out = pd.concat([frost_resistant_01["frost_resitant_01"], frost_resistant_001["frost_resitant_001"], form["form_1"], f1_vibration, rms, flatness, rolloff, contrast, f1_audio, c, centroid, Q], axis=1).reset_index()
out = out.fillna("")

print(out)

   sample_name frost_resitant_01 frost_resitant_001 form_1  f1_vibration  \
0          S01               1.0                0.0     pl   2748.000000   
1          S02               1.0                0.0     pl   3289.000000   
2          S03               1.0                0.0     pl   1665.211513   
3          S04               1.0                0.0     pt   5750.000000   
4          S05               1.0                0.0     pt   5372.000000   
5          S06               1.0                0.0     pt   5804.000000   
6          S07               1.0                0.0     pt   5447.000000   
7          S08               1.0                1.0     pt   3370.000000   
8          S09               1.0                1.0     pt   3622.000000   
9          S10               1.0                0.0     pt   3115.000000   
10         S11               0.0                0.0     pl   2014.886014   
11         S12               0.0                0.0     pl   1531.468471   
12         S

In [ ]:
out

,sample_name,frost_resitant_01,frost_resitant_001,form_1,f1_vibration,rms_1,rms_2,rms_3,rms_4,rms_5,rms_6,rms_7,rms_8,rms_9,flatness_1,flatness_2,flatness_3,flatness_4,flatness_5,flatness_6,flatness_7,flatness_8,flatness_9,rolloff_1,rolloff_2,rolloff_3,rolloff_4,rolloff_5,rolloff_6,rolloff_7,rolloff_8,rolloff_9,contrast_1,contrast_2,contrast_3,contrast_4,contrast_5,contrast_6,contrast_7,contrast_8,contrast_9,f1_audio_1,f1_audio_2,f1_audio_3,f1_audio_4,f1_audio_5,f1_audio_6,f1_audio_7,f1_audio_8,f1_audio_9,c_1,c_2,c_3,c_4,c_5,c_6,c_7,c_8,c_9,centroid_1,centroid_2,centroid_3,centroid_4,centroid_5,centroid_6,centroid_7,centroid_8,centroid_9,Q_1,Q_2,Q_3,Q_4,Q_5,Q_6,Q_7,Q_8,Q_9
0,S01,1.0,0.0,pl,2748.000000,0.122204,0.129421,0.111234,0.128124,0.130879,0.114371,0.101053,0.122191,0.103574,0.008963,0.007643,0.012436,0.008403,0.01339,0.014131,0.012329,0.007448,0.002556,12397.290432,14052.751905,16177.053345,13538.27265,16525.57155,14973.835732,10887.044877,10633.954276,6397.798476,14.673802,10.566528,12.863068,14.208599,19.863773,13.529413,13.348872,16.384968,15.057705,3223.793395,3227.942422,3223.793395,3227.942422,3227.942422,3227.942422,3223.793395,3223.793395,3223.793395,55.140871,59.485896,57.89086,63.182687,60.316329,66.255863,59.274832,64.133336,58.61262,5841.668799,6028.62343,6446.748089,6010.692139,6893.026751,6581.526085,5742.024542,5688.223022,4881.061628,183.579095,170.388946,174.858539,160.419565,168.043038,152.978751,170.775874,157.838526,172.70532
1,S02,1.0,0.0,pl,3289.000000,0.118736,0.145612,0.109484,0.137154,0.138425,0.13242,0.109408,0.147778,0.134464,0.016339,0.015454,0.006394,0.015358,0.027054,0.011955,0.00697,0.007724,0.006021,18280.609653,18268.162574,10787.468247,17600.169348,18741.151566,16089.923793,8679.762913,16152.159187,10376.714649,29.826742,18.81998,18.723538,18.68509,30.014621,17.509814,11.260597,20.458865,12.831292,3315.071973,3593.056732,3315.071973,3593.056732,3593.056732,3593.056732,3315.071973,3593.056732,3315.071973,53.590213,55.768092,52.344184,52.534777,56.854063,54.397365,52.550665,48.540497,56.479053,7814.978773,7963.651772,5625.709233,7481.775787,8567.668763,7020.566623,5358.211364,6684.951085,5688.567321,194.239311,202.305613,198.863085,214.756752,198.441371,207.403396,198.081716,232.428568,184.304188
2,S03,1.0,0.0,pl,1665.211513,0.105056,0.118863,0.093317,0.106551,0.100342,0.09387,0.106784,0.0855,0.091838,0.00994,0.009023,0.006666,0.006019,0.006953,0.002398,0.012777,0.012875,0.005959,13911.685013,15558.848434,6837.595258,6879.085521,7609.314141,4742.337003,16214.394581,13650.296359,5895.7663,15.193014,17.11473,14.292372,10.734138,21.940421,8.499996,14.247616,17.561312,5.571778,3900.084674,1788.230313,3900.084674,3900.084674,1788.230313,3900.084674,1788.230313,1788.230313,3895.935648,56.806662,44.97485,59.088341,51.434448,58.481212,62.897749,44.583643,54.349873,55.834852,6197.090933,5959.802284,5035.394561,5013.24937,5459.173114,4558.928434,6193.474734,6016.156455,4705.888714,215.577989,124.848513,207.253506,238.094629,96.01448,194.701178,125.944018,103.312904,219.096809
3,S04,1.0,0.0,pt,5750.000000,0.103743,0.107076,0.106194,,,,,,,0.03596,0.016435,0.009913,,,,,,,18550.296359,15231.07536,11185.774767,,,,,,,13.438899,12.800314,8.528069,,,,,,,5750.550381,5750.550381,5750.550381,,,,,,,104.057411,103.448224,96.625126,,,,,,,9268.764887,8021.454829,7279.460418,,,,,,,173.526595,174.54846,186.874045,,,,,,
4,S05,1.0,0.0,pt,5372.000000,0.106428,0.103779,0.109662,,,,,,,0.013317,0.009995,0.005318,,,,,,,13795.512278,10716.934801,7285.690093,,,,,,,19.972425,8.571825,8.611889,,,,,,,5368.839966,5368.839966,5368.839966,,,,,,,95.872463,95.976631,86.70923,,,,,,,7666.6358,7107.350129,6278.363827,,,,,,,175.839411,175.648565,194.421719,,,,,,
5,S06,1.0,0.0,pt,5804.000000,0.102041,0.102716,0.102248,,,,,,,0.010802,0.00821,0.009769,,,,,,,10712.785775,8576.037257,9999.15326,,,,,,,9.612124,9.810438,12.262222,,,,,,,5808.636749,5808.636749,5808.636749,,,,,,,110.161899,104.201227,103.432419,,,,,,,7280.09518,6869.146299,7196

In [ ]:
stats = out.copy()

# Pop column in string format
sample_name = stats.pop('sample_name')
form_1 = stats.pop('form_1')

# Convert other columns in float
stats = stats.apply(pd.to_numeric)

# Compute stats (mean, std, min, max) for each features

rms_columns = ['rms_1', 'rms_2', 'rms_3', 'rms_4', 'rms_5', 'rms_6', 'rms_7', 'rms_8', 'rms_9']
stats['rms_mean'] = stats[rms_columns].mean(axis=1)
stats['rms_std'] = stats[rms_columns].std(axis=1)
stats['rms_min'] = stats[rms_columns].min(axis=1)
stats['rms_max'] = stats[rms_columns].max(axis=1)

flatness_columns = ['flatness_1', 'flatness_2', 'flatness_3', 'flatness_4', 'flatness_5', 'flatness_6', 'flatness_7', 'flatness_8', 'flatness_9']
stats['flatness_mean'] = stats[flatness_columns].mean(axis=1)
stats['flatness_std'] = stats[flatness_columns].std(axis=1)
stats['flatness_min'] = stats[flatness_columns].min(axis=1)
stats['flatness_max'] = stats[flatness_columns].max(axis=1)

rolloff_columns = ['rolloff_1', 'rolloff_2', 'rolloff_3', 'rolloff_4', 'rolloff_5', 'rolloff_6', 'rolloff_7', 'rolloff_8', 'rolloff_9']
stats['rolloff_mean'] = stats[rolloff_columns].mean(axis=1)
stats['rolloff_std'] = stats[rolloff_columns].std(axis=1)
stats['rolloff_min'] = stats[rolloff_columns].min(axis=1)
stats['rolloff_max'] = stats[rolloff_columns].max(axis=1)

contrast_columns = ['contrast_1', 'contrast_2', 'contrast_3', 'contrast_4', 'contrast_5', 'contrast_6', 'contrast_7', 'contrast_8', 'contrast_9']
stats['contrast_mean'] = stats[contrast_columns].mean(axis=1)
stats['contrast_std'] = stats[contrast_columns].std(axis=1)
stats['contrast_min'] = stats[contrast_columns].min(axis=1)
stats['contrast_max'] = stats[contrast_columns].max(axis=1)

f1_audio_columns = ['f1_audio_1', 'f1_audio_2', 'f1_audio_3', 'f1_audio_4', 'f1_audio_5', 'f1_audio_6', 'f1_audio_7', 'f1_audio_8', 'f1_audio_9']
stats['f1_audio_mean'] = stats[f1_audio_columns].mean(axis=1)
stats['f1_audio_std'] = stats[f1_audio_columns].std(axis=1)
stats['f1_audio_min'] = stats[f1_audio_columns].min(axis=1)
stats['f1_audio_max'] = stats[f1_audio_columns].max(axis=1)

centroid_columns = ['centroid_1', 'centroid_2', 'centroid_3', 'centroid_4', 'centroid_5', 'centroid_6', 'centroid_7', 'centroid_8', 'centroid_9']
stats['centroid_mean'] = stats[centroid_columns].mean(axis=1)
stats['centroid_std'] = stats[centroid_columns].std(axis=1)
stats['centroid_min'] = stats[centroid_columns].min(axis=1)
stats['centroid_max'] = stats[centroid_columns].max(axis=1)

c_columns = ['c_1', 'c_2', 'c_3', 'c_4', 'c_5', 'c_6', 'c_7', 'c_8', 'c_9']
stats['c_mean'] = stats[c_columns].mean(axis=1)
stats['c_std'] = stats[c_columns].std(axis=1)
stats['c_min'] = stats[c_columns].min(axis=1)
stats['c_max'] = stats[c_columns].max(axis=1)

Q_columns = ['Q_1', 'Q_2', 'Q_3', 'Q_4', 'Q_5', 'Q_6', 'Q_7', 'Q_8', 'Q_9']
stats['Q_mean'] = stats[Q_columns].mean(axis=1)
stats['Q_std'] = stats[Q_columns].std(axis=1)
stats['Q_min'] = stats[Q_columns].min(axis=1)
stats['Q_max'] = stats[Q_columns].max(axis=1)


#out[rms_columns].dtypes

stats.insert(0, 'form_1', form_1)
stats.insert(0, 'sample_name', sample_name)


stats

,sample_name,form_1,frost_resitant_01,frost_resitant_001,f1_vibration,rms_1,rms_2,rms_3,rms_4,rms_5,rms_6,rms_7,rms_8,rms_9,flatness_1,flatness_2,flatness_3,flatness_4,flatness_5,flatness_6,flatness_7,flatness_8,flatness_9,rolloff_1,rolloff_2,rolloff_3,rolloff_4,rolloff_5,rolloff_6,rolloff_7,rolloff_8,rolloff_9,contrast_1,contrast_2,contrast_3,contrast_4,contrast_5,contrast_6,contrast_7,contrast_8,contrast_9,f1_audio_1,f1_audio_2,f1_audio_3,f1_audio_4,f1_audio_5,f1_audio_6,f1_audio_7,f1_audio_8,f1_audio_9,c_1,c_2,c_3,c_4,c_5,c_6,c_7,c_8,c_9,centroid_1,centroid_2,centroid_3,centroid_4,centroid_5,centroid_6,centroid_7,centroid_8,centroid_9,Q_1,Q_2,Q_3,Q_4,Q_5,Q_6,Q_7,Q_8,Q_9,rms_mean,rms_std,rms_min,rms_max,flatness_mean,flatness_std,flatness_min,flatness_max,rolloff_mean,rolloff_std,rolloff_min,rolloff_max,contrast_mean,contrast_std,contrast_min,contrast_max,f1_audio_mean,f1_audio_std,f1_audio_min,f1_audio_max,centroid_mean,centroid_std,centroid_min,centroid_max,c_mean,c_std,c_min,c_max,Q_mean,Q_std,Q_min,Q_max
0,S01,pl,1.0,0.0,2748.000000,0.122204,0.129421,0.111234,0.128124,0.130879,0.114371,0.101053,0.122191,0.103574,0.008963,0.007643,0.012436,0.008403,0.013390,0.014131,0.012329,0.007448,0.002556,12397.290432,14052.751905,16177.053345,13538.272650,16525.571550,14973.835732,10887.044877,10633.954276,6397.798476,14.673802,10.566528,12.863068,14.208599,19.863773,13.529413,13.348872,16.384968,15.057705,3223.793395,3227.942422,3223.793395,3227.942422,3227.942422,3227.942422,3223.793395,3223.793395,3223.793395,55.140871,59.485896,57.890860,63.182687,60.316329,66.255863,59.274832,64.133336,58.612620,5841.668799,6028.623430,6446.748089,6010.692139,6893.026751,6581.526085,5742.024542,5688.223022,4881.061628,183.579095,170.388946,174.858539,160.419565,168.043038,152.978751,170.775874,157.838526,172.705320,0.118117,0.011114,0.101053,0.130879,0.009700,0.003712,0.002556,0.014131,12842.619249,3193.930404,6397.798476,16525.571550,14.499636,2.577989,10.566528,19.863773,3225.637407,2.186729e+00,3223.793395,3227.942422,6012.621609,589.024741,4881.061628,6893.026751,60.477033,3.449210,55.140871,66.255863,167.954184,9.431595,152.978751,183.579095
1,S02,pl,1.0,0.0,3289.000000,0.118736,0.145612,0.109484,0.137154,0.138425,0.132420,0.109408,0.147778,0.134464,0.016339,0.015454,0.006394,0.015358,0.027054,0.011955,0.006970,0.007724,0.006021,18280.609653,18268.162574,10787.468247,17600.169348,18741.151566,16089.923793,8679.762913,16152.159187,10376.714649,29.826742,18.819980,18.723538,18.685090,30.014621,17.509814,11.260597,20.458865,12.831292,3315.071973,3593.056732,3315.071973,3593.056732,3593.056732,3593.056732,3315.071973,3593.056732,3315.071973,53.590213,55.768092,52.344184,52.534777,56.854063,54.397365,52.550665,48.540497,56.479053,7814.978773,7963.651772,5625.709233,7481.775787,8567.668763,7020.566623,5358.211364,6684.951085,5688.567321,194.239311,202.305613,198.863085,214.756752,198.441371,207.403396,198.081716,232.428568,184.304188,0.130387,0.014484,0.109408,0.147778,0.012585,0.006859,0.006021,0.027054,14997.346881,3933.530719,8679.762913,18741.151566,19.792282,6.474943,11.260597,30.014621,3469.507950,1.465108e+02,3315.071973,3593.056732,6911.786747,1151.963200,5358.211364,8567.668763,53.673212,2.587895,48.540497,56.854063,203.424889,13.742620,184.304188,232.428568
2,S03,pl,1.0,0.0,1665.211513,0.105056,0.118863,0.093317,0.106551,0.100342,0.093870,0.106784,0.085500,0.091838,0.009940,0.009023,0.006666,0.006019,0.006953,0.002398,0.012777,0.012875,0.005959,13911.685013,15558.848434,6837.595258,6879.085521,7609.314141,4742.337003,16214.394581,13650.296359,5895.766300,15.193014,17.114730,14.292372,10.734138,21.940421,8.499996,14.247616,17.561312,5.571778,3900.084674,1788.230313,3900.084674,3900.084674,1788.230313,3900.084674,1788.230313,1788.230313,3895.935648,56.806662,44.974850,59.088341,51.434448,58.481212,62.897749,44.583643,54.349873,55.834852,6197.090933,5959.802284,5035.394561,5013.249370,5459.173114,4558.928434,6193.474734,6016.156455,4705.88

In [ ]:
print(list(stats.columns.values))

['sample_name', 'form_1', 'frost_resitant_01', 'frost_resitant_001', 'f1_vibration', 'rms_1', 'rms_2', 'rms_3', 'rms_4', 'rms_5', 'rms_6', 'rms_7', 'rms_8', 'rms_9', 'flatness_1', 'flatness_2', 'flatness_3', 'flatness_4', 'flatness_5', 'flatness_6', 'flatness_7', 'flatness_8', 'flatness_9', 'rolloff_1', 'rolloff_2', 'rolloff_3', 'rolloff_4', 'rolloff_5', 'rolloff_6', 'rolloff_7', 'rolloff_8', 'rolloff_9', 'contrast_1', 'contrast_2', 'contrast_3', 'contrast_4', 'contrast_5', 'contrast_6', 'contrast_7', 'contrast_8', 'contrast_9', 'f1_audio_1', 'f1_audio_2', 'f1_audio_3', 'f1_audio_4', 'f1_audio_5', 'f1_audio_6', 'f1_audio_7', 'f1_audio_8', 'f1_audio_9', 'c_1', 'c_2', 'c_3', 'c_4', 'c_5', 'c_6', 'c_7', 'c_8', 'c_9', 'centroid_1', 'centroid_2', 'centroid_3', 'centroid_4', 'centroid_5', 'centroid_6', 'centroid_7', 'centroid_8', 'centroid_9', 'Q_1', 'Q_2', 'Q_3', 'Q_4', 'Q_5', 'Q_6', 'Q_7', 'Q_8', 'Q_9', 'rms_mean', 'rms_std', 'rms_min', 'rms_max', 'flatness_mean', 'flatness_std', 'flatness

In [ ]:
# Exporter en CSV sans l’index
stats.to_csv('/content/drive/MyDrive/Article sound/stats.csv', index=False, encoding='utf-8')